# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

It is the Docker version of [`Jemula802_Setup.ipynb`](Jemula802_Setup.ipynb). Both leave you with a working Jemula. They differ in what you have to install:

| | [`Jemula802_Setup.ipynb`](Jemula802_Setup.ipynb) | this notebook |
| --- | --- | --- |
| needs on your machine | a JDK, 17 or newer | Docker Desktop |
| how Jemula is built | `javac` into `simulator/kernel/bin` and `simulator/engine/bin` | once, inside a container image |

The image carries Java 18, Gradle 8.7 and Python 3.10, so Docker is the only thing you install. It is the same on Windows, macOS and Linux: Docker Desktop runs a small Linux virtual machine on Windows and macOS, and the container is Linux either way.

Start Docker Desktop before running the cells below.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import os
import pathlib
import platform
import subprocess
import sys

**Declarations**

Set the variables below, then run the rest of the notebook from top to bottom.

- `PROJECT_DIR` is the Jemula project, relative to this notebook. It is the same repository [`Jemula802_Setup.ipynb`](Jemula802_Setup.ipynb) clones, with a `Dockerfile` at its top level.
- `REPO_URL` is only used when `PROJECT_DIR` is not there yet. If you unpacked a release archive instead, put the folder next to this notebook and leave the URL alone.
- `IMAGE` is the name given to the image built from the project.

In [ ]:
# --- Declarations: editable variables for students ---
PROJECT_DIR = "jemula-project"
REPO_URL = "https://gitlab.ethz.ch/wireless/jemula.git"
IMAGE = "jemula:latest"

PROJECT_PATH = pathlib.Path(PROJECT_DIR).resolve()

print(f"Host      : {platform.system()} {platform.machine()}")
print(f"Project   : {PROJECT_PATH}")
print(f"Image     : {IMAGE}")

### 1. Jemula Setup

#### 1.1. Checking Docker

Everything below runs through Docker, so this fails early and clearly if the engine is not reachable. The usual cause is that Docker Desktop is installed but not started.

In [ ]:
# --- Check that the Docker engine is running ---
try:
    info = subprocess.run(
        ["docker", "info", "--format", "{{.ServerVersion}}|{{.OSType}}/{{.Architecture}}"],
        check=True, capture_output=True, text=True).stdout.strip()
    version, arch = info.split("|")
    print(f"Docker engine {version} is running, building for {arch}.")
except FileNotFoundError:
    raise SystemExit("Docker is not installed, or the `docker` command is not on PATH.")
except subprocess.CalledProcessError:
    raise SystemExit("Docker is installed but the engine is not reachable. "
                     "Start Docker Desktop and run this cell again.")

#### 1.2. Getting the Project

Two ways to have the project on disk, and this accepts either:

- you unpacked a release archive into `PROJECT_DIR`, in which case it is used as it is;
- or `REPO_URL` is set and `PROJECT_DIR` is missing, in which case it is cloned.

A folder that is a git checkout is updated with `git pull`. A folder that is not is left alone, because it did not come from git and pulling would fail.

In [ ]:
# --- Find or fetch the project ---
if PROJECT_PATH.is_dir() and (PROJECT_PATH / ".git").is_dir() and REPO_URL:
    print(f"Updating {PROJECT_DIR} ...")
    subprocess.run(["git", "-C", str(PROJECT_PATH), "pull", "--ff-only"], check=True)
    print(f"{PROJECT_DIR} updated.")
elif PROJECT_PATH.is_dir():
    print(f"Using the project already in {PROJECT_DIR}.")
elif REPO_URL:
    print(f"Cloning into {PROJECT_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_PATH)], check=True)
    print(f"{PROJECT_DIR} cloned.")
else:
    raise SystemExit(f"No project at {PROJECT_PATH}, and REPO_URL is empty. "
                     f"Set REPO_URL to the repository, or clone it there by hand.")

if not (PROJECT_PATH / "Dockerfile").is_file():
    raise SystemExit(f"No Dockerfile in {PROJECT_PATH}. This is probably one of the two "
                     f"original repositories rather than the combined project.")
print("Dockerfile found, the project looks right.")

#### 1.3. Building the Image

The first build takes a few minutes, most of it installing the JDK and JupyterLab. Later builds reuse the cached layers and finish in seconds.

The build compiles the simulator, so a checkout that does not compile cannot produce an image.

In [ ]:
# --- Build the image (skip if it already exists) ---
REBUILD = False   # set True to force a rebuild

existing = subprocess.run(["docker", "images", "-q", IMAGE],
                          capture_output=True, text=True).stdout.strip()

if existing and not REBUILD:
    print(f"Image {IMAGE} already exists. Set REBUILD = True to build it again.")
else:
    print(f"Building {IMAGE}, this takes a few minutes the first time ...")
    build = subprocess.Popen(["docker", "build", "-t", IMAGE, str(PROJECT_PATH)],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in build.stdout:
        print(line, end="")
    build.wait()
    if build.returncode != 0:
        raise SystemExit(f"Build failed with exit code {build.returncode}.")
    print(f"\n{IMAGE} built.")

#### 1.4. What is in the Image

A quick look at what you just got, so it is clear that none of it had to be installed on your machine.

In [ ]:
# --- Show the toolchain inside the image ---
report = subprocess.run(
    ["docker", "run", "--rm", "--entrypoint", "bash", IMAGE, "-lc",
     "java -version 2>&1 | head -1; ./gradlew --version 2>/dev/null | grep -E '^Gradle'; "
     "python3 --version; echo; echo 'scenarios:'; ls my-simulations/scenarios/*/ | head -5"],
    capture_output=True, text=True).stdout
print(report)

### 2. Running Jemula

#### 2.1. Running a Scenario

This folder, with the project in it, is mounted into the container: scenarios are read from your disk, and the results are written next to this notebook, in `results/`, where the other notebooks look for them. Your algorithms in `my-simulations/algorithms/` are compiled before every run.

Scenario paths are relative to this folder, the same as in [`Jemula802_Run.ipynb`](Jemula802_Run.ipynb), which also runs scenarios in this image once it is built.

In [ ]:
# --- Run one scenario inside the container ---
def run_scenario(scenario, memory="2G"):
    """Run a scenario in the container and stream its output.

    `scenario` is relative to this folder, for example
    "./jemula-project/my-simulations/scenarios/examples/Jemula802GettingStarted.xml".
    Results are written next to this notebook, as with Jemula802_Run.ipynb.
    """
    lib = "/workspace/simulator/engine/build/install/engine/lib/*"
    algorithms = f"{PROJECT_DIR}/my-simulations/algorithms/layer2_802Algorithms/*.java"
    process = subprocess.Popen(
        ["docker", "run", "--rm", "-v", f"{os.getcwd()}:/work", "-w", "/work",
         "--entrypoint", "bash", IMAGE, "-lc",
         f"mkdir -p /tmp/alg && javac -nowarn -encoding UTF-8 -d /tmp/alg -cp '{lib}' {algorithms} "
         f"&& java -Xms{memory} -Xmx{memory} -cp '/tmp/alg:{lib}' emulator.JE802Starter '{scenario}'"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    return process.returncode


scenario = "./jemula-project/my-simulations/scenarios/examples/Jemula802GettingStarted.xml"
print(f"Running example scenario {scenario} ...")
exit_code = run_scenario(scenario)
print(f"\nExample scenario finished with exit code {exit_code}")

#### 2.2. Where the Results Went

Next to this notebook, in the folder the scenario's `Path2Results` names (under `results/`). [`Jemula802_Results.ipynb`](Jemula802_Results.ipynb) reads them from there.

In [ ]:
# --- List what the run produced ---
import xml.etree.ElementTree as ET

results_rel = ET.parse(scenario).getroot().find(".//JE802StatEval").get("Path2Results")
results_path = pathlib.Path(results_rel)

print(f"{results_path.resolve()}\n")
for item in sorted(results_path.iterdir())[:10]:
    print(f"  {item.name}")

#### 2.3. Jemula in the Browser

To use Jemula in your browser - pick a scenario, change its settings, press Run and watch its log and packet timeline - start its console from [`Jemula802_Run.ipynb`](Jemula802_Run.ipynb), section 1.3. It runs in the image you built above.